<a href="https://colab.research.google.com/github/profliuhao/CSIT599/blob/main/CSIT599_Online_lab4_sentiment_analysis_rnn_lstm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 — Simple RNN vs. LSTM for Sentiment Analysis on IMDB Reviews (Keras)

**Module 4: Introduction to Natural Language Processing**

In this lab you will build and compare **two recurrent models** in Keras that decide
whether a movie review is **positive or negative**, using the classic **IMDB** dataset
(25,000 labeled reviews):

```
review (word ids) --> [Embedding] --> [SimpleRNN or LSTM] --> [Dropout] --> [Dense] --> P(positive)
```

The comparison is the point: the **Simple RNN** carries information through a single
hidden state and suffers from vanishing gradients, while the **LSTM** adds gating
(input / forget / output gates) that protects long-range information. You will measure
what that difference is worth in accuracy and training time.

## What you will do
Fill in each `___BLANK___` (a hint comment sits next to every one):
  1. choose the **sequence length** used for padding/truncation
  2. complete the **Simple RNN** model
  3. complete the **LSTM** model
  4. pick the **loss function and optimizer**
  5. set the **training hyperparameters** (batch size, epochs, validation split)

Everything else — data loading, early stopping, evaluation, plots, and the comparison
table — is already written for you.

## How to use this file
* Recommended: **Google Colab** with a GPU runtime (`Runtime > Change runtime type > T4 GPU`).
  CPU works too — a few minutes per model.
* Fill in each `___BLANK___`, then run the cells from top to bottom (or "Run All").

## Setup — imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, SimpleRNN, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import time

# Set random seeds for reproducibility
np.random.seed(42)
import tensorflow as tf
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU available:", len(tf.config.list_physical_devices("GPU")) > 0)

## STEP 1 — Load and preprocess the data

Each IMDB review arrives as a sequence of **integer word ids** (1 = most frequent word).
Reviews have different lengths, so we pad/truncate them all to `maxlen` tokens to form
rectangular batches.

In [ ]:
print("\n" + "="*70)
print("STEP 1: Loading IMDB Dataset")
print("="*70)

# We'll use only the top 10,000 most frequent words
max_features = 10000  # Vocabulary size

# ___BLANK___: How many words to consider per review?
# Hint: Reviews will be truncated or padded to this length. Try values like 200, 300, or 500.
maxlen = ___BLANK___  # Try values like 200, 300, or 500

print(f"Loading IMDB dataset with vocabulary size: {max_features}")
print(f"Maximum sequence length: {maxlen}")

# x_train / x_test are lists of word-id sequences; y_train / y_test are 0/1 labels.
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)

print(f"\nTraining samples: {len(x_train)}")
print(f"Test samples: {len(x_test)}")
print(f"Example review length before padding: {len(x_train[0])}")
print(f"Example label: {y_train[0]} (1 = positive, 0 = negative)")

# Pad sequences: shorter reviews get 0s, longer reviews are truncated.
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)

print(f"\nAfter padding:")
print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")

## STEP 2 — Build the Simple RNN model

Architecture: `Embedding → SimpleRNN → Dropout → Dense(1, sigmoid)`.

In [ ]:
print("\n" + "="*70)
print("STEP 2: Building Simple RNN Model")
print("="*70)

def build_simple_rnn_model():
    """
    Build a Simple RNN model for sentiment classification.

    1. Embedding layer: converts word ids to dense 128-d vectors
    2. SimpleRNN layer: processes the sequence step by step
    3. Dropout layer: prevents overfitting
    4. Dense output layer: one sigmoid unit for binary classification
    """
    model = Sequential([
        Input(shape=(maxlen,)),
        # (batch, maxlen) -> (batch, maxlen, 128)
        Embedding(input_dim=max_features, output_dim=128),

        # ___BLANK___: Add the SimpleRNN layer.
        # Hint: use an appropriate number of units (try 64 or 128) and
        # return_sequences=False since we only need the final output.
        SimpleRNN(units=___BLANK___, return_sequences=False),

        Dropout(0.5),
        Dense(1, activation="sigmoid"),
    ])
    return model

rnn_model = build_simple_rnn_model()
rnn_model.summary()

## STEP 3 — Build the LSTM model

Identical, except the recurrent layer is an **LSTM**. Its gates (input, forget, output)
control what enters, stays in, and leaves the memory cell — the fix for vanishing
gradients you read about this module.

In [ ]:
print("\n" + "="*70)
print("STEP 3: Building LSTM Model")
print("="*70)

def build_lstm_model():
    """
    Build an LSTM model for sentiment classification.

    LSTM advantages over Simple RNN:
    - Better at learning long-term dependencies
    - Mitigates the vanishing gradient problem
    - Uses gates (input, forget, output) to control information flow
    """
    model = Sequential([
        Input(shape=(maxlen,)),
        Embedding(input_dim=max_features, output_dim=128),

        # ___BLANK___: Add the LSTM layer.
        # Hint: use the same number of units as the SimpleRNN so the comparison is fair;
        # return_sequences=False for sequence classification.
        LSTM(units=___BLANK___, return_sequences=False),

        Dropout(0.5),
        Dense(1, activation="sigmoid"),
    ])
    return model

lstm_model = build_lstm_model()
lstm_model.summary()

## STEP 4 — Compile both models

Same loss, optimizer, and metric for both — the only difference we want to measure is
the recurrent layer itself.

In [ ]:
print("\n" + "="*70)
print("STEP 4: Compiling Models")
print("="*70)

# ___BLANK___: Specify the loss function and optimizer.
# Hints:
# - For binary classification with a sigmoid output, use 'binary_crossentropy'.
# - Popular optimizers: 'adam', 'rmsprop', 'sgd'.
loss_function = '___BLANK___'
optimizer = '___BLANK___'

print(f"Loss function: {loss_function}")
print(f"Optimizer: {optimizer}")

rnn_model.compile(loss=loss_function, optimizer=optimizer, metrics=["accuracy"])
lstm_model.compile(loss=loss_function, optimizer=optimizer, metrics=["accuracy"])

print("✓ Models compiled successfully")

## STEP 5 — Train both models

Early stopping watches the validation loss and restores the best weights, so a few
extra epochs can't hurt the final model.

In [ ]:
print("\n" + "="*70)
print("STEP 5: Training Models")
print("="*70)

# ___BLANK___: Set the training hyperparameters.
# Hints:
# - batch_size: typically 32, 64, or 128
# - epochs: typically 5-15 for this dataset (early stopping may end training sooner)
# - validation_split: typically 0.2 (20% of the training data)
batch_size = ___BLANK___
epochs = ___BLANK___
validation_split = ___BLANK___

print(f"Batch size: {batch_size}")
print(f"Epochs: {epochs}")
print(f"Validation split: {validation_split}")

# Stop when validation loss hasn't improved for 2 epochs; keep the best weights.
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

print("\n" + "-"*70)
print("Training Simple RNN Model...")
print("-"*70)
start_time = time.time()
rnn_history = rnn_model.fit(
    x_train, y_train,
    batch_size=batch_size, epochs=epochs, validation_split=validation_split,
    callbacks=[early_stop], verbose=1,
)
rnn_training_time = time.time() - start_time
print(f"✓ Simple RNN trained in {rnn_training_time:.2f} seconds")

print("\n" + "-"*70)
print("Training LSTM Model...")
print("-"*70)
start_time = time.time()
lstm_history = lstm_model.fit(
    x_train, y_train,
    batch_size=batch_size, epochs=epochs, validation_split=validation_split,
    callbacks=[early_stop], verbose=1,
)
lstm_training_time = time.time() - start_time
print(f"✓ LSTM trained in {lstm_training_time:.2f} seconds")

## STEP 6 — Evaluate on the test set *(provided)*

In [ ]:
print("\n" + "="*70)
print("STEP 6: Evaluating Models on Test Set")
print("="*70)

rnn_loss, rnn_accuracy = rnn_model.evaluate(x_test, y_test, verbose=0)
print(f"\nSimple RNN Test Results:")
print(f"  Loss: {rnn_loss:.4f}")
print(f"  Accuracy: {rnn_accuracy:.4f} ({rnn_accuracy*100:.2f}%)")
print(f"  Training time: {rnn_training_time:.2f} seconds")

lstm_loss, lstm_accuracy = lstm_model.evaluate(x_test, y_test, verbose=0)
print(f"\nLSTM Test Results:")
print(f"  Loss: {lstm_loss:.4f}")
print(f"  Accuracy: {lstm_accuracy:.4f} ({lstm_accuracy*100:.2f}%)")
print(f"  Training time: {lstm_training_time:.2f} seconds")

## STEP 7 — Visualize training history *(provided)*

In [ ]:
print("\n" + "="*70)
print("STEP 7: Visualizing Training History")
print("="*70)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(rnn_history.history["accuracy"], label="RNN Train", marker="o")
axes[0].plot(rnn_history.history["val_accuracy"], label="RNN Val", marker="o")
axes[0].plot(lstm_history.history["accuracy"], label="LSTM Train", marker="s")
axes[0].plot(lstm_history.history["val_accuracy"], label="LSTM Val", marker="s")
axes[0].set_title("Model Accuracy Comparison", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(rnn_history.history["loss"], label="RNN Train", marker="o")
axes[1].plot(rnn_history.history["val_loss"], label="RNN Val", marker="o")
axes[1].plot(lstm_history.history["loss"], label="LSTM Train", marker="s")
axes[1].plot(lstm_history.history["val_loss"], label="LSTM Val", marker="s")
axes[1].set_title("Model Loss Comparison", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## STEP 8 — Summary and comparison *(provided)*

In [ ]:
print("\n" + "="*70)
print("STEP 8: Final Comparison Summary")
print("="*70)

print("\n" + "-"*70)
print("PERFORMANCE COMPARISON")
print("-"*70)
print(f"{'Metric':<20} {'Simple RNN':<15} {'LSTM':<15} {'Winner':<15}")
print("-"*70)
print(f"{'Test Accuracy':<20} {rnn_accuracy:.4f}{'':<10} {lstm_accuracy:.4f}{'':<10} {'LSTM' if lstm_accuracy > rnn_accuracy else 'RNN':<15}")
print(f"{'Test Loss':<20} {rnn_loss:.4f}{'':<10} {lstm_loss:.4f}{'':<10} {'LSTM' if lstm_loss < rnn_loss else 'RNN':<15}")
print(f"{'Training Time (s)':<20} {rnn_training_time:.2f}{'':<10} {lstm_training_time:.2f}{'':<10} {'RNN' if rnn_training_time < lstm_training_time else 'LSTM':<15}")
print("-"*70)

accuracy_improvement = ((lstm_accuracy - rnn_accuracy) / rnn_accuracy) * 100
print(f"\nLSTM accuracy improvement over RNN: {accuracy_improvement:+.2f}%")

In [ ]:
print("\n" + "="*70)
print("KEY TAKEAWAYS")
print("="*70)
print("""
1. LSTM ADVANTAGES:
   - Better at capturing long-term dependencies in sequences
   - More resistant to the vanishing gradient problem
   - Uses gating mechanisms (input, forget, output gates)
   - Generally achieves higher accuracy on sequence tasks

2. SIMPLE RNN LIMITATIONS:
   - Struggles with long sequences due to vanishing gradients
   - Limited memory of past information
   - Simpler architecture, fewer parameters

3. TRADE-OFFS:
   - LSTM: higher accuracy but slower training and more parameters
   - Simple RNN: faster training but lower performance on complex tasks

4. WHEN TO USE EACH:
   - Use LSTM when: long-term dependencies matter, accuracy is critical
   - Use Simple RNN when: sequences are short, speed is critical, or as a baseline
""")

## Checklist before you submit

* [ ] All cells run top-to-bottom without errors, outputs visible.
* [ ] Both models train, and early stopping restores the best weights.
* [ ] The LSTM reaches at least **~85%** test accuracy (the Simple RNN is typically lower).
* [ ] The accuracy/loss comparison plots and the summary table are shown.
* [ ] In a final markdown cell (2-4 sentences): which model won, by how much, and what
      mechanism explains the gap? Bring these numbers to the M4 Discussion.

**Submit your completed notebook (.ipynb) with all outputs visible.**